In [ ]:
import ipywidgets as widgets
import joblib as jl
import pandas as pd
from IPython.display import HTML, display

In [ ]:
best_params_from_GA = jl.load("models/best_params_from_GA.pkl")
print(f"Loaded commodities: {len(best_params_from_GA)}")

In [ ]:
commodities = sorted(best_params_from_GA.keys())
first_commodity = commodities[0]
horizons = sorted(best_params_from_GA[first_commodity].keys())

commodity_dropdown = widgets.Dropdown(
    options=commodities,
    value=first_commodity,
    description="Komoditas:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="60%"),
)

horizon_dropdown = widgets.Dropdown(
    options=horizons,
    value=horizons[0],
    description="Horizon (hari):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="40%"),
)

output = widgets.Output()


def refresh_horizon_options(change):
    selected_commodity = change["new"]
    available_horizons = sorted(best_params_from_GA[selected_commodity].keys())
    horizon_dropdown.options = available_horizons
    horizon_dropdown.value = available_horizons[0]


def render(_=None):
    with output:
        output.clear_output()

        commodity = commodity_dropdown.value
        horizon = horizon_dropdown.value
        params = best_params_from_GA[commodity][horizon]

        df = pd.DataFrame(params.items(), columns=["Hyperparameter", "Value"])
        df["Value"] = df["Value"].apply(
            lambda v: f"{v:.6f}" if isinstance(v, float) else str(v)
        )

        title = f"<h4 style='margin: 0 0 8px 0;'>Best Params | {commodity} | h={horizon}</h4>"
        display(HTML(title))
        display(df)


commodity_dropdown.observe(refresh_horizon_options, names="value")
commodity_dropdown.observe(render, names="value")
horizon_dropdown.observe(render, names="value")

render()
display(widgets.HBox([commodity_dropdown, horizon_dropdown]))
display(output)